In [33]:
import os
import pandas as pd

In [35]:
df_path = "defects4j.csv"
new_df_path = 'standardized_defects4j.csv'
processed_df_path = 'processed/defects4j.csv'
os.makedirs('processed', exist_ok=True)

In [37]:
def standardize_df(df_path, new_df_path):
    column_names = ['source_code', 'vuln_lines']
    df = pd.read_csv(df_path, header=None, names=column_names)
    df['index'] = range(0, len(df))  # Starts from 0
    df = df[['index'] + df.columns[:-1].tolist()]  # move 'index' column to front
    df.to_csv(new_df_path, index=False)

In [39]:
standardize_df(df_path, new_df_path)

In [41]:
def map_target_lines(code, target_lines):
    # Split the code into lines.
    original_lines = code.splitlines()
    new_lines = []
    mapping = {}  # mapping from original line number to new line number

    # Process each original line (1-indexed).
    for i, line in enumerate(original_lines, start=1):
        if line.strip():  # if the line is not empty (or only whitespace)
            new_lines.append(line)
            mapping[i] = len(new_lines)  # record new line number
        else:
            mapping[i] = None  # line is removed (empty)

    # For each target line, get the corresponding new line number.
    new_target_lines = {}
    for orig in target_lines:
        new_num = mapping.get(orig)
        if orig <= len(original_lines):
            if new_num is not None:
                new_target_lines[orig] = new_num
            else:
                # Optionally, you might want to handle cases where the target line is empty.
                new_target_lines[orig] = "Removed (empty line)"
    return new_lines, new_target_lines

In [43]:
df = pd.read_csv(new_df_path)
new_rows = []
item_index = 0
for i, row in df.iterrows():
    source_code = row['source_code']
    index = row['index']
    vuln_lines = eval(row['vuln_lines'])
    new_lines, new_target_lines = map_target_lines(source_code, vuln_lines)
    # print(new_target_lines)
    new_source_code = '\n'.join(new_lines)

    new_line_numbers_for_new_code = []
    for orig, new in new_target_lines.items():
        if isinstance(new, int):
            new_line_numbers_for_new_code.append(new)
        else:
            # print("Something wrong with line mapping after code compression by removing lines with whitespaces", new)
            continue
    new_line_numbers_for_new_code.sort()

    zero_indexed_target_lines = []
    for e in new_line_numbers_for_new_code:
        if e - 1 >= 0:
            zero_indexed_target_lines.append(e - 1)
        else:
            raise Exception("Non-continuable error occurred!!!") 

    if len(new_line_numbers_for_new_code) > 0:
        row = {'item_index': item_index, 'source_code': new_source_code, 'vuln_lines': str(zero_indexed_target_lines)}
        new_rows.append(row)
        item_index += 1
    else:
        pass
        # print(f"Something wrong with the source_code in index {index}")
new_df = pd.DataFrame(new_rows)
new_df.to_csv('processed/defects4j.csv', index=False)

In [44]:
new_df

,item_index,source_code,vuln_lines
0,0,* are not present.\n */\n private ...,[9]
1,1,}\n // Native constructors are alwa...,"[34, 35, 47, 53, 60, 61, 76, 81, 86, 96, 97, 9..."
2,2,public void notAMockPassedToVerify(Class t...,"[9, 18, 19, 26, 30, 34, 67, 75, 83, 91, 96, 10..."
3,3,"AbstractCompiler compiler, Node root, Ca...",[87]
4,4,sb.append(quote);\n return sb.toString(...,"[24, 26, 28]"
...,...,...,...
1265,1265,/* Helper methods for populating method in...,[9]
1266,1266,charsetName = foundCharset;\n ...,[11]
1267,1267,* <p>No delimiter is added before or afte...,"[55, 56, 59, 60, 61, 73]"
1268,1268,package com.fasterxml.jackson.databind;\nimpor...,"[18, 21, 39, 48, 53, 73]"
